# Phase 5 - Multivariate LSTM: Primary Forecasting Model

This is the centrepiece of the entire project. I will build, train, and evaluate a **Multivariate LSTM with Attention**; a deep learning model that ingests 60 days of market history across 12 features and forecasts VIX at three horizons: **t+1, t+5, and t+10 trading days ahead**. The primary evaluation target is **t+5 (one week ahead)**.

### Baselines to Beat (from Phase 4)
| Model | MAE (VIX pts) | Directional Accuracy |
|---|---|---|
| Persistence (Naive) | 2.2295 | 53.03% |
| ARIMA(1,1,1) | 2.2580 | 53.55% |

> **Note:** ARIMA failed to beat the naive persistence baseline. A model that simply says 'VIX in 5 days = VIX today' outperformed classical time series. This is the strongest possible motivation for a multivariate LSTM as it has access to 12 features including macro data and sentiment that ARIMA never sees.

## Cell 1 - Plotting Backend, Imports & Reproducibility Seeds

In [1]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)


import os, sys, json, random, warnings
warnings.filterwarnings('ignore')


import numpy as np
import pandas as pd


import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

os.makedirs('models',  exist_ok=True)
os.makedirs('figures', exist_ok=True)
os.makedirs('logs',    exist_ok=True)


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')
print(f'Seed    : {SEED} — every run will produce identical results')
print(f'CWD     : {os.getcwd()}')
print('All imports OK')

Device  : cpu
PyTorch : 2.10.0+cpu
Seed    : 42 — every run will produce identical results
CWD     : c:\Users\sabin\genai-market
All imports OK


## Cell 2 - Hyperparameters

**Why these specific values (result of 4 tuning runs):**
- `SEQUENCE_LENGTH = 60` - 60 trading days = one full market quarter. Gives the model enough context to see developing trends, regime transitions, and macro shifts. 30 days was too short.
- `HIDDEN_SIZE = 128` - doubles capacity vs the base plan's 64. With 12 features and 6,577 rows this is still a small model (~120K params) that trains in under 20 minutes on CPU.
- `DROPOUT = 0.2` - regularises the larger model without being so aggressive it prevents learning
- `BATCH_SIZE = 32` - smaller than the base plan's 64. Smaller batches introduce gradient noise that helps escape shallow local minima and improves generalisation on financial data
- `WEIGHT_DECAY = 1e-4` - L2 regularisation penalty on large weights, works alongside dropout
- `HORIZON_WEIGHTS` - t+5 is our primary output so it contributes 60% of the loss. t+1 and t+10 each contribute 20%. The model is optimised for what we actually care about.
- `PATIENCE = 20` - gives the model enough room after a plateau for the scheduler to reduce LR and find further improvements before stopping

In [3]:
#Data splits 
TRAIN_END = '2018-12-31'
VAL_END   = '2022-12-31'

# Model architecture 
SEQUENCE_LENGTH = 60
HORIZONS        = [1, 5, 10]      # t+1, t+5, t+10 trading days ahead
HIDDEN_SIZE     = 128
NUM_LAYERS      = 2
DROPOUT         = 0.2

# Training 
BATCH_SIZE      = 32
LEARNING_RATE   = 1e-3
WEIGHT_DECAY    = 1e-4
MAX_EPOCHS      = 200
PATIENCE        = 20
SCHED_PATIENCE  = 8

#  Weighted loss — t+5 is primary so it gets 60% weight 
HORIZON_WEIGHTS = torch.tensor([0.2, 0.6, 0.2], dtype=torch.float32)

print('Hyperparameters confirmed:')
print(f'  Sequence length  : {SEQUENCE_LENGTH} trading days (1 quarter)')
print(f'  Horizons         : {HORIZONS} days  weights={HORIZON_WEIGHTS.tolist()}')
print(f'  Hidden size      : {HIDDEN_SIZE}')
print(f'  Num layers       : {NUM_LAYERS}')
print(f'  Dropout          : {DROPOUT}')
print(f'  Batch size       : {BATCH_SIZE}')
print(f'  Learning rate    : {LEARNING_RATE}')
print(f'  Weight decay     : {WEIGHT_DECAY}')
print(f'  Max epochs       : {MAX_EPOCHS}')
print(f'  Patience         : {PATIENCE}')
print(f'  Sched patience   : {SCHED_PATIENCE}')

Hyperparameters confirmed:
  Sequence length  : 60 trading days (1 quarter)
  Horizons         : [1, 5, 10] days  weights=[0.20000000298023224, 0.6000000238418579, 0.20000000298023224]
  Hidden size      : 128
  Num layers       : 2
  Dropout          : 0.2
  Batch size       : 32
  Learning rate    : 0.001
  Weight decay     : 0.0001
  Max epochs       : 200
  Patience         : 20
  Sched patience   : 8


## Cell 3 - Load Data & Create Time-Ordered Splits

I will load the single master dataset produced by Phase 3 (`data/processed/master_df.csv`) and split it strictly by calendar date.

**Why never random split a time series:**  
If row 500 (year 2002) ends up in the test set while row 501 (the very next trading day) is in training, the model has effectively seen the future during training. This produces falsely optimistic results that completely collapse in production.

**Feature columns:**  
I will exclude `regime_label` and `sentiment_source` (categorical strings), `vix` (raw level - will use the log version), and `vix_return` (pct change used by GARCH in Phase 4). The 12 remaining numeric columns are the input feature matrix the LSTM will learn from.

In [4]:
df = pd.read_csv('data/processed/master_df.csv', index_col=0, parse_dates=True)

# Time-ordered splits — NEVER random 
train_df = df[df.index <= TRAIN_END].copy()
val_df   = df[(df.index > TRAIN_END) & (df.index <= VAL_END)].copy()
test_df  = df[df.index > VAL_END].copy()

# Feature columns 
EXCLUDE      = {'regime_label', 'sentiment_source', 'vix', 'vix_log', 'vix_return'}
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE]
N_FEATURES   = len(FEATURE_COLS)

print(f'Master df shape : {df.shape}')
print(f'Date range      : {df.index.min().date()} to {df.index.max().date()}')
print(f'NaN values      : {df.isnull().sum().sum()}')
print()
print(f'Train : {len(train_df):,} rows  ({train_df.index.min().date()} → {train_df.index.max().date()})')
print(f'Val   : {len(val_df):,}   rows  ({val_df.index.min().date()} → {val_df.index.max().date()})')
print(f'Test  : {len(test_df):,}   rows  ({test_df.index.min().date()} → {test_df.index.max().date()})')
print()
print(f'Features ({N_FEATURES}): {FEATURE_COLS}')
print(f'Target          : vix_log → back-transformed via np.exp() for all evaluation')
print()
print('Regime distribution (full dataset):')
for regime, count in df['regime_label'].value_counts().items():
    pct = count / len(df) * 100
    print(f'  {regime:<10}: {count:,} days ({pct:.1f}%)')

Master df shape : (6577, 17)
Date range      : 2000-02-02 to 2026-03-27
NaN values      : 0

Train : 4,758 rows  (2000-02-02 → 2018-12-31)
Val   : 1,008   rows  (2019-01-02 → 2022-12-30)
Test  : 811   rows  (2023-01-03 → 2026-03-27)

Features (12): ['vix_lag1', 'vix_lag5', 'vix_lag21', 'vix_roll_mean5', 'vix_roll_std21', 'fedfunds', 'cpi', 'unrate', 'gs10', 'indpro', 'm2sl', 'sentiment']
Target          : vix_log → back-transformed via np.exp() for all evaluation

Regime distribution (full dataset):
  LOW       : 4,056 days (61.7%)
  ELEVATED  : 1,900 days (28.9%)
  CRISIS    : 621 days (9.4%)


## Cell 4 - PyTorch Dataset & DataLoaders

**VIXDataset - sliding window construction:**  
The `VIXDataset` class converts the flat time series into overlapping 60-day windows. Each sample contains:
- **X**: shape `(60, 12)` - 60 consecutive trading days × 12 features
- **y**: shape `(3,)` - log-VIX values at t+1, t+5, and t+10

**Why sliding windows?**  
LSTMs process sequences. A single row of master_df has no temporal context on its own. By packaging 60 consecutive rows as one training sample, the LSTM can learn patterns like: "VIX has been climbing for 6 weeks, sentiment is negative, and fedfunds is rising - forecast a continued spike."

**Why `shuffle=False`:**  
This is a time series. Shuffling would mix samples from different time periods within a batch - a sample from 2005 next to one from 2018. The LSTM must see data in the order it happened. `shuffle=False` preserves strict chronological order across all batches.

**Why `drop_last=False`:**  
With no shuffling, `drop_last=True` would silently discard the final incomplete batch - real training samples from the end of the time series that the model never sees. We keep all samples.

In [5]:
class VIXDataset(Dataset):
    """
    Sliding-window time series dataset for VIX forecasting.

    Each sample:
        X   : (sequence_length, n_features) - input window, chronological order
        y   : (n_horizons,)                 - log-VIX at t+1, t+5, t+10
        idx : integer position in the dataframe (for regime lookup in analysis)
    """
    def __init__(self, dataframe, feature_cols, target_col='vix_log',
                 sequence_length=60, horizons=None):
        if horizons is None:
            horizons = [5]
        self.seq_len  = sequence_length
        self.horizons = horizons
        self.max_h    = max(horizons)
        self.X        = dataframe[feature_cols].values.astype(np.float32)
        self.target   = dataframe[target_col].values.astype(np.float32)
        # Valid indices: need seq_len history AND max_h future rows
        self.indices  = list(range(sequence_length, len(dataframe) - self.max_h))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        x   = self.X[idx - self.seq_len : idx]                              # (seq_len, n_features)
        y   = np.array([self.target[idx + h] for h in self.horizons],
                       dtype=np.float32)                                     # (n_horizons,)
        return torch.tensor(x), torch.tensor(y), idx


# datasets 
train_ds = VIXDataset(train_df, FEATURE_COLS, horizons=HORIZONS, sequence_length=SEQUENCE_LENGTH)
val_ds   = VIXDataset(val_df,   FEATURE_COLS, horizons=HORIZONS, sequence_length=SEQUENCE_LENGTH)
test_ds  = VIXDataset(test_df,  FEATURE_COLS, horizons=HORIZONS, sequence_length=SEQUENCE_LENGTH)

# DataLoaders — shuffle=False to preserve time order 
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

#  Verifying shapes 
xb, yb, _ = next(iter(train_loader))
print(f'Train samples : {len(train_ds):,}')
print(f'Val samples   : {len(val_ds):,}')
print(f'Test samples  : {len(test_ds):,}')
print()
print(f'Batch X shape : {tuple(xb.shape)}  → (batch_size, seq_len=60, n_features=12)')
print(f'Batch y shape : {tuple(yb.shape)}  → (batch_size, n_horizons=3)')
print()
print('shuffle=False  - data stays in strict chronological order')
print('drop_last=False - no training samples discarded')

Train samples : 4,688
Val samples   : 938
Test samples  : 741

Batch X shape : (32, 60, 12)  → (batch_size, seq_len=60, n_features=12)
Batch y shape : (32, 3)  → (batch_size, n_horizons=3)

shuffle=False  - data stays in strict chronological order
drop_last=False - no training samples discarded


## Cell 5 - Model Architecture: LSTM + Attention

### Base: 2-Layer LSTM
The LSTM processes the 60-day input sequence step by step, maintaining a hidden state that accumulates temporal context. Two stacked LSTM layers allow the model to learn hierarchical temporal patterns - the first layer captures short-term dynamics, the second captures longer-term structure.

###  Additive Attention Mechanism
Standard LSTM uses only the **final** hidden state for prediction implicitly weighting the most recent timestep most. Our attention layer instead computes a score for **all 60 timesteps** and takes a weighted sum of all hidden states.

This means:
- A VIX spike 6 weeks ago may be more informative than yesterday's calm day - attention can learn this
- Phase 7 (Explainability) can visualise attention weights to show *which days* drove the forecast
- Adds only ~130 parameters (negligible cost, meaningful expressiveness gain)

### Multi-Horizon Output Head
One linear layer maps `hidden_size → 3 outputs` (t+1, t+5, t+10). Training on all three simultaneously forces the model to learn representations useful across multiple time scales - it cannot overfit to just one horizon. The t+5 output is our primary evaluation metric.

In [6]:
class Attention(nn.Module):
    """
    Additive (Bahdanau-style) self-attention over LSTM output sequence.

    Returns:
        context : (batch, hidden_size) - weighted sum of all hidden states
        weights : (batch, seq_len)     - attention weights for interpretability
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.attn    = nn.Linear(hidden_size, hidden_size)
        self.context = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, lstm_output):
        # lstm_output: (batch, seq_len, hidden_size)
        energy  = self.context(torch.tanh(self.attn(lstm_output)))  # (batch, seq_len, 1)
        weights = torch.softmax(energy, dim=1)                       # (batch, seq_len, 1)
        context = (weights * lstm_output).sum(dim=1)                 # (batch, hidden_size)
        return context, weights.squeeze(-1)                          # weights: (batch, seq_len)


class LSTMModel(nn.Module):
    """
    Multivariate LSTM + Attention for multi-horizon VIX forecasting.

    Args:
        input_size  : number of input features (12)
        hidden_size : LSTM hidden units per layer (128)
        num_layers  : stacked LSTM layers (2)
        dropout     : dropout between LSTM layers and before output (0.2)
        n_horizons  : number of forecast horizons (3)
    """
    def __init__(self, input_size, hidden_size=128, num_layers=2,
                 dropout=0.2, n_horizons=3):
        super().__init__()
        self.lstm      = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.attention = Attention(hidden_size)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_size, n_horizons)

    def forward(self, x):
        lstm_out, _     = self.lstm(x)              # (batch, seq_len, hidden)
        context, attn_w = self.attention(lstm_out)  # (batch, hidden), (batch, seq_len)
        out             = self.fc(self.dropout(context))  # (batch, n_horizons)
        return out, attn_w


# Instantiate 
model = LSTMModel(
    input_size  = N_FEATURES,
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    dropout     = DROPOUT,
    n_horizons  = len(HORIZONS)
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

#  Sanity check: forward pass with dummy input 
with torch.no_grad():
    dummy_x         = torch.zeros(2, SEQUENCE_LENGTH, N_FEATURES).to(DEVICE)
    dummy_out, dummy_w = model(dummy_x)

print(model)
print()
print(f'Trainable parameters : {n_params:,}')
print(f'Output shape         : {tuple(dummy_out.shape)}  → (batch=2, n_horizons=3)')
print(f'Attention shape      : {tuple(dummy_w.shape)}  → (batch=2, seq_len=60)')
print(f'Device               : {DEVICE}')
print('Model OK')

LSTMModel(
  (lstm): LSTM(12, 128, num_layers=2, batch_first=True, dropout=0.2)
  (attention): Attention(
    (attn): Linear(in_features=128, out_features=128, bias=True)
    (context): Linear(in_features=128, out_features=1, bias=False)
  )
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)

Trainable parameters : 221,827
Output shape         : (2, 3)  → (batch=2, n_horizons=3)
Attention shape      : (2, 60)  → (batch=2, seq_len=60)
Device               : cpu
Model OK


## Cell 6 - Training Setup: Loss, Optimiser, Scheduler, Helpers

###  Weighted Loss Function
Instead of treating t+1, t+5, and t+10 equally, I will weight the loss so t+5 (our primary target) contributes 60% of the gradient signal. t+1 and t+10 each contribute 20%. This means the model is explicitly optimised for what we care about most, while still benefiting from the regularising effect of multi-horizon training.

###  Correct Directional Accuracy
The standard implementation of directional accuracy uses `np.diff()` on consecutive predictions; this measures whether prediction[t] is in the same direction as prediction[t-1], which is meaningless for a forecasting model. The correct metric asks: **did the model correctly predict whether VIX will be higher or lower than it is today?** Our `compute_metrics` function computes this properly by comparing `pred[i]` vs `current_vix[i]` against `true[i]` vs `current_vix[i]`.

###  Gradient Norm Logging
I will clip gradients at `max_norm=1.0` to prevent exploding gradients, a common problem when training on financial time series that contain extreme events like COVID March 2020 (VIX 82.7). The `run_epoch` function now returns the mean gradient norm so we can see in the training printout whether clipping is actually firing.

In [7]:
#  Move horizon weights to device 
hw = HORIZON_WEIGHTS.to(DEVICE)  # [0.2, 0.6, 0.2]


def weighted_mse_loss(preds, targets):
    """
    Addition 3: Weighted MSE - t+5 gets 60% of the loss signal.
    preds, targets : (batch, n_horizons)
    """
    sq_errors = (preds - targets) ** 2          # (batch, n_horizons)
    weighted  = sq_errors * hw.unsqueeze(0)      # broadcast weights across batch
    return weighted.mean()


optimizer = torch.optim.Adam(
    model.parameters(),
    lr           = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=SCHED_PATIENCE
)


def run_epoch(loader, training=True):
    """
    One full pass through a DataLoader.
    Returns (mean_loss, mean_grad_norm) — grad_norm=0.0 during eval.
    Addition 6: gradient norm tracked and returned for logging.
    """
    model.train() if training else model.eval()
    total_loss  = 0.0
    total_gnorm = 0.0
    n_batches   = 0

    with torch.set_grad_enabled(training):
        for xb, yb, _ in loader:
            xb, yb   = xb.to(DEVICE), yb.to(DEVICE)
            preds, _ = model(xb)
            loss     = weighted_mse_loss(preds, yb)

            if training:
                optimizer.zero_grad()
                loss.backward()
                gnorm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                total_gnorm += gnorm.item()
                optimizer.step()

            total_loss += loss.item() * len(xb)
            n_batches  += 1

    mean_loss  = total_loss / len(loader.dataset)
    mean_gnorm = total_gnorm / n_batches if training else 0.0
    return mean_loss, mean_gnorm


def compute_metrics(preds_vix, true_vix, current_vix):
    """
    Addition 5: Correct directional accuracy.
    All inputs already in VIX space (back-transformed via np.exp()).

    Directional accuracy = did the model correctly forecast whether
    VIX will be HIGHER or LOWER than today's VIX?
    This is the meaningful metric for a t+5 forecasting model.
    """
    mae     = float(np.mean(np.abs(preds_vix - true_vix)))
    rmse    = float(np.sqrt(np.mean((preds_vix - true_vix) ** 2)))
    mape    = float(np.mean(np.abs((preds_vix - true_vix) / true_vix)) * 100)

    # Correct direction: forecast direction vs today, not vs yesterday's forecast
    pred_direction = np.sign(preds_vix  - current_vix)
    true_direction = np.sign(true_vix   - current_vix)
    dir_acc = float(np.mean(pred_direction == true_direction) * 100)

    return {'mae': mae, 'rmse': rmse, 'mape': mape, 'dir_acc': dir_acc}


print('Loss      : Weighted MSE  weights=[0.2, 0.6, 0.2]  (t+1, t+5, t+10)')
print('Optimiser : Adam  lr=1e-3  weight_decay=1e-4')
print('Scheduler : ReduceLROnPlateau  factor=0.5  patience=8')
print('Clipping  : max_norm=1.0 - gradient norm logged each epoch')
print('Dir Acc   : forecast direction vs today (correct implementation)')
print('Setup OK')

Loss      : Weighted MSE  weights=[0.2, 0.6, 0.2]  (t+1, t+5, t+10)
Optimiser : Adam  lr=1e-3  weight_decay=1e-4
Scheduler : ReduceLROnPlateau  factor=0.5  patience=8
Clipping  : max_norm=1.0 - gradient norm logged each epoch
Dir Acc   : forecast direction vs today (correct implementation)
Setup OK


## Cell 7 - Training Loop

In [7]:
train_losses     = []
val_losses       = []
grad_norms       = []
best_val_loss    = float('inf')
best_epoch       = 0
patience_counter = 0
best_state       = None

print(f'Training up to {MAX_EPOCHS} epochs  |  patience={PATIENCE}  |  sched_patience={SCHED_PATIENCE}')
print(f'Device  : {DEVICE}  |  batch={BATCH_SIZE}  |  seq={SEQUENCE_LENGTH}  |  hidden={HIDDEN_SIZE}')
print(f'Loss    : Weighted MSE  t+5 weight=0.6')
print('─' * 78)
print(f'{"Epoch":>6}  {"Train":>9}  {"Val":>9}  {"Best":>9}  {"BestEp":>6}  {"LR":>9}  {"GNorm":>7}  {"Pat":>6}')
print('─' * 78)

for epoch in range(1, MAX_EPOCHS + 1):

    tr_loss, gnorm = run_epoch(train_loader, training=True)
    val_loss, _    = run_epoch(val_loader,   training=False)
    scheduler.step(val_loss)

    train_losses.append(tr_loss)
    val_losses.append(val_loss)
    grad_norms.append(gnorm)

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_epoch       = epoch
        patience_counter = 0
        best_state       = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if epoch % 5 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'{epoch:>6}  {tr_loss:>9.5f}  {val_loss:>9.5f}  '
              f'{best_val_loss:>9.5f}  {best_epoch:>6}  '
              f'{lr_now:>9.2e}  {gnorm:>7.4f}  {patience_counter:>3}/{PATIENCE}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping triggered at epoch {epoch}')
        break

# ── Restore best weights ──────────────────────────────────────────────────────
model.load_state_dict(best_state)
model.eval()

gap = val_losses[best_epoch - 1] - train_losses[best_epoch - 1]

if best_epoch < 15:
    diagnosis = 'UNDERFITTING  -best epoch too early, model needs more capacity or longer training'
elif gap > 0.06:
    diagnosis = 'OVERFITTING - large train/val gap, increase dropout or reduce hidden_size'
elif gap > 0.03:
    diagnosis = 'MILD OVERFITTING - acceptable for financial time series, monitor in Phase 6'
else:
    diagnosis = 'GOOD FIT —-train and val losses are well aligned'

print(f'\n{"─" * 78}')
print(f'Training complete')
print(f'  Best epoch       : {best_epoch}')
print(f'  Best val loss    : {best_val_loss:.6f}')
print(f'  Total epochs run : {len(train_losses)}')
print(f'  Train/val gap    : {gap:.6f}  (at best epoch)')
print(f'  Mean grad norm   : {np.mean(grad_norms):.4f}')
print(f'  Diagnosis        : {diagnosis}')

Training up to 200 epochs  |  patience=20  |  sched_patience=8
Device  : cpu  |  batch=32  |  seq=60  |  hidden=128
Loss    : Weighted MSE  t+5 weight=0.6
──────────────────────────────────────────────────────────────────────────────
 Epoch      Train        Val       Best  BestEp         LR    GNorm     Pat
──────────────────────────────────────────────────────────────────────────────
     1    0.18333    0.20309    0.20309       1   1.00e-03   1.3457    0/20
     5    0.06717    0.15178    0.15178       5   1.00e-03   1.1690    0/20
    10    0.06128    0.16176    0.15178       5   1.00e-03   1.1400    5/20
    15    0.08364    0.12607    0.12607      15   5.00e-04   1.4411    0/20
    20    0.07346    0.12144    0.12099      17   5.00e-04   1.3326    3/20
    25    0.07433    0.12380    0.11696      21   5.00e-04   1.3499    4/20
    30    0.07276    0.11841    0.11696      21   2.50e-04   1.3298    9/20
    35    0.05695    0.09703    0.09398      34   2.50e-04   1.1103    1/20
   

## Cell 8 - Save Model & Config

In [8]:
# Save weights
torch.save(model.state_dict(), 'models/lstm_model.pt')

#  Save full config for Phase 9 reconstruction 
config = {
    # Architecture — needed to reconstruct model in Phase 9
    'input_size'      : N_FEATURES,
    'hidden_size'     : HIDDEN_SIZE,
    'num_layers'      : NUM_LAYERS,
    'dropout'         : DROPOUT,
    'n_horizons'      : len(HORIZONS),
    'horizons'        : HORIZONS,
    'horizon_weights' : HORIZON_WEIGHTS.tolist(),
    'sequence_length' : SEQUENCE_LENGTH,
    'feature_cols'    : FEATURE_COLS,
    'target_col'      : 'vix_log',
    # Training metadata
    'best_epoch'      : best_epoch,
    'best_val_loss'   : best_val_loss,
    'total_epochs'    : len(train_losses),
    'mean_grad_norm'  : float(np.mean(grad_norms)),
    'diagnosis'       : diagnosis,
    'seed'            : SEED,
    'train_end'       : TRAIN_END,
    'val_end'         : VAL_END,
    'n_params'        : n_params,
    'has_attention'   : True,
}

with open('models/lstm_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Saved: models/lstm_model.pt')
print('Saved: models/lstm_config.json')
print()
print(json.dumps(config, indent=2))

Saved: models/lstm_model.pt
Saved: models/lstm_config.json

{
  "input_size": 12,
  "hidden_size": 128,
  "num_layers": 2,
  "dropout": 0.2,
  "n_horizons": 3,
  "horizons": [
    1,
    5,
    10
  ],
  "horizon_weights": [
    0.20000000298023224,
    0.6000000238418579,
    0.20000000298023224
  ],
  "sequence_length": 60,
  "feature_cols": [
    "vix_lag1",
    "vix_lag5",
    "vix_lag21",
    "vix_roll_mean5",
    "vix_roll_std21",
    "fedfunds",
    "cpi",
    "unrate",
    "gs10",
    "indpro",
    "m2sl",
    "sentiment"
  ],
  "target_col": "vix_log",
  "best_epoch": 96,
  "best_val_loss": 0.030319334522112093,
  "total_epochs": 116,
  "mean_grad_norm": 0.9229282648907767,
  "diagnosis": "GOOD FIT \u2014-train and val losses are well aligned",
  "seed": 42,
  "train_end": "2018-12-31",
  "val_end": "2022-12-31",
  "n_params": 221827,
  "has_attention": true
}


In [8]:
# Reload saved model  skip retraining
with open('models/lstm_config.json') as f:
    config = json.load(f)

model = LSTMModel(
    input_size  = config['input_size'],
    hidden_size = config['hidden_size'],
    num_layers  = config['num_layers'],
    dropout     = config['dropout'],
    n_horizons  = config['n_horizons']
).to(DEVICE)

model.load_state_dict(torch.load('models/lstm_model.pt', map_location=DEVICE))
model.eval()

best_epoch    = config['best_epoch']
best_val_loss = config['best_val_loss']
diagnosis     = config['diagnosis']
train_losses  = [0.0] * config['total_epochs']
val_losses    = [0.0] * config['total_epochs']
grad_norms    = [config['mean_grad_norm']] * config['total_epochs']

print(f'Model loaded from disk')
print(f'Best epoch     : {best_epoch}')
print(f'Best val loss  : {best_val_loss:.6f}')
print(f'Diagnosis      : {diagnosis}')

Model loaded from disk
Best epoch     : 96
Best val loss  : 0.030319
Diagnosis      : GOOD FIT —-train and val losses are well aligned


## Cell 9 - Training Curves Plot

In [10]:
import gc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
gc.collect()

epochs_list = list(range(1, len(train_losses) + 1))
zoom_start  = max(0, len(train_losses) - 30)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Train vs Val Loss', 'Last 30 Epochs'])

fig.add_trace(go.Scatter(x=epochs_list, y=train_losses,
              name='Train', line=dict(color='steelblue')), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_list, y=val_losses,
              name='Val', line=dict(color='tomato')), row=1, col=1)
fig.add_vline(x=best_epoch, line_dash='dash', line_color='green',
              annotation_text=f'Best epoch {best_epoch}', row=1, col=1)

fig.add_trace(go.Scatter(x=epochs_list[zoom_start:], y=train_losses[zoom_start:],
              name='Train zoom', line=dict(color='steelblue'),
              showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs_list[zoom_start:], y=val_losses[zoom_start:],
              name='Val zoom', line=dict(color='tomato'),
              showlegend=False), row=1, col=2)

fig.update_layout(title=f'LSTM Training Curves - {diagnosis}', height=420)
fig.write_image('figures/lstm_training_curves.png', width=1200, height=420, scale=1)
del fig
gc.collect()
print('Saved: figures/lstm_training_curves.png')

Saved: figures/lstm_training_curves.png


## Cell 10 - Test Set Evaluation

Running inference on the held-out test set (2023–present). This data was never seen during training or hyperparameter tuning.

**Dual-target analysis:**  
All metrics are computed in VIX space after `np.exp()` back-transform. We also report the raw log-space error to show the amplification effect - VIX-space MAE is always higher because `np.exp()` expands the error. Training on log(VIX) produces a smoother loss landscape because crisis spikes (VIX 80+) are compressed to ~4.4 in log space instead of dominating the gradient.

**Primary comparison:** t+5 MAE vs the Phase 4 persistence baseline of **2.2295 VIX points**.


In [12]:
model.eval()

all_preds   = []
all_targets = []
all_attn    = []

with torch.no_grad():
    for xb, yb, _ in test_loader:
        xb          = xb.to(DEVICE)
        out, attn_w = model(xb)
        all_preds.append(out.cpu().numpy())
        all_targets.append(yb.numpy())
        all_attn.append(attn_w.cpu().numpy())

preds_log    = np.vstack(all_preds)    # (n_test, 3) — log-VIX space
targets_log  = np.vstack(all_targets)  # (n_test, 3)
attn_weights = np.vstack(all_attn)     # (n_test, 60)

#  back-transform to VIX space 
preds_vix   = np.exp(preds_log)    # (n_test, 3)
targets_vix = np.exp(targets_log)  # (n_test, 3)

# Current VIX (today's value) for correct directional accuracy
current_vix_arr = test_df['vix'].iloc[SEQUENCE_LENGTH : SEQUENCE_LENGTH + len(preds_vix)].values

print('=' * 65)
print('LSTM TEST SET RESULTS — Multi-Horizon Evaluation')
print('=' * 65)

results       = {}
horizon_labels = ['t+1  (1-day)', 't+5  (1-week) ← PRIMARY', 't+10 (2-week)']

for i, (h, label) in enumerate(zip(HORIZONS, horizon_labels)):
    m = compute_metrics(preds_vix[:, i], targets_vix[:, i], current_vix_arr)
    results[h] = m
    print(f'\n  [{label}]')
    print(f'    MAE              : {m["mae"]:.4f} VIX pts')
    print(f'    RMSE             : {m["rmse"]:.4f} VIX pts')
    print(f'    MAPE             : {m["mape"]:.2f}%')
    print(f'    Directional Acc  : {m["dir_acc"]:.2f}%  (vs today, correct implementation)')

#  log vs VIX space error comparison 
log_mae_5 = float(np.mean(np.abs(preds_log[:, 1] - targets_log[:, 1])))
vix_mae_5 = results[5]['mae']

print(f'\n{"─" * 65}')
print(f'Addition 2 — Log vs VIX space error (t+5):')
print(f'  Log-space MAE  : {log_mae_5:.6f}  (training target space)')
print(f'  VIX-space MAE  : {vix_mae_5:.4f}   (after np.exp() back-transform)')
print(f'  Amplification  : {vix_mae_5/log_mae_5:.1f}x  (expected — log compresses crisis spikes)')

#  Primary comparison 
mae5 = results[5]['mae']
dir5 = results[5]['dir_acc']
beat = mae5 < 2.2295

print(f'\n{"═" * 65}')
print(f'Persistence baseline  : MAE=2.2295  Dir=53.03%')
print(f'LSTM (t+5)            : MAE={mae5:.4f}  Dir={dir5:.2f}%')
print(f'Beat persistence      : {"YES ✓" if beat else "NO ✗ - see Phase 6 for conformal intervals"}')
print(f'{"═" * 65}')

LSTM TEST SET RESULTS — Multi-Horizon Evaluation

  [t+1  (1-day)]
    MAE              : 4.2536 VIX pts
    RMSE             : 5.2666 VIX pts
    MAPE             : 26.12%
    Directional Acc  : 49.93%  (vs today, correct implementation)

  [t+5  (1-week) ← PRIMARY]
    MAE              : 4.3079 VIX pts
    RMSE             : 5.3682 VIX pts
    MAPE             : 26.39%
    Directional Acc  : 54.79%  (vs today, correct implementation)

  [t+10 (2-week)]
    MAE              : 4.3431 VIX pts
    RMSE             : 5.4664 VIX pts
    MAPE             : 26.51%
    Directional Acc  : 58.30%  (vs today, correct implementation)

─────────────────────────────────────────────────────────────────
Addition 2 — Log vs VIX space error (t+5):
  Log-space MAE  : 0.231315  (training target space)
  VIX-space MAE  : 4.3079   (after np.exp() back-transform)
  Amplification  : 18.6x  (expected — log compresses crisis spikes)

═════════════════════════════════════════════════════════════════
Persistence

## Cell 11 - Predictions vs Actual Plot

In [14]:
import gc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
gc.collect()

n           = len(preds_vix)
test_dates  = test_df.index[SEQUENCE_LENGTH : SEQUENCE_LENGTH + n]
regime_arr  = test_df['regime_label'].iloc[SEQUENCE_LENGTH : SEQUENCE_LENGTH + n].values
n           = min(n, len(test_dates))
actual_5    = targets_vix[:n, 1]
predicted_5 = preds_vix[:n, 1]
residuals   = predicted_5 - actual_5
dates       = test_dates[:n]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Actual vs Predicted VIX (t+5)',
                                    'Residuals (Predicted - Actual)'],
                    vertical_spacing=0.12)

fig.add_trace(go.Scatter(x=dates, y=actual_5, name='Actual VIX',
              line=dict(color='black', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=dates, y=predicted_5, name='LSTM Pred (t+5)',
              line=dict(color='steelblue', width=1.2, dash='dash')), row=1, col=1)
fig.add_hline(y=20, line_dash='dot', line_color='orange', row=1, col=1)
fig.add_hline(y=30, line_dash='dot', line_color='red',    row=1, col=1)

colors_r = ['steelblue' if r >= 0 else 'tomato' for r in residuals]
fig.add_trace(go.Bar(x=dates, y=list(residuals),
              marker_color=colors_r, name='Residual',
              showlegend=False), row=2, col=1)
fig.add_hline(y=0, line_color='black', row=2, col=1)

fig.update_yaxes(title_text='VIX',      row=1, col=1)
fig.update_yaxes(title_text='Residual', row=2, col=1)
fig.update_layout(title='LSTM t+5 Predictions vs Actual - Test Set', height=600)
fig.write_image('figures/lstm_predictions_vs_actual.png', width=1200, height=600, scale=1)
del fig
gc.collect()
print('Saved: figures/lstm_predictions_vs_actual.png')
print(f'Mean residual: {residuals.mean():+.4f} VIX pts')

Saved: figures/lstm_predictions_vs_actual.png
Mean residual: +3.0029 VIX pts


## Cell 12 - Residual Analysis by Regime


In [15]:
import gc
import plotly.graph_objects as go
gc.collect()

print('=' * 65)
print('RESIDUAL ANALYSIS BY REGIME (t+5)')
print('=' * 65)

regime_summary = {}
palette = {'LOW': 'steelblue', 'ELEVATED': 'orange', 'CRISIS': 'crimson'}

fig = go.Figure()

for regime, color in palette.items():
    mask = regime_arr[:n] == regime
    if mask.sum() == 0:
        continue
    p    = predicted_5[mask]
    t    = actual_5[mask]
    mae  = float(np.mean(np.abs(p - t)))
    rmse = float(np.sqrt(np.mean((p - t) ** 2)))
    bias = float(np.mean(p - t))
    regime_summary[regime] = {'mae': mae, 'rmse': rmse, 'bias': bias, 'n': int(mask.sum())}
    print(f'  {regime:<10}: n={mask.sum():>4} | MAE={mae:.4f} | RMSE={rmse:.4f} | Bias={bias:+.4f}')

    fig.add_trace(go.Histogram(
        x=(p - t).tolist(),
        name=f'{regime} (n={mask.sum()})',
        marker_color=color,
        opacity=0.55,
        nbinsx=35,
        histnorm='probability density'
    ))

if 'CRISIS' in regime_summary and 'LOW' in regime_summary:
    ratio = regime_summary['CRISIS']['mae'] / regime_summary['LOW']['mae']
    print(f'\n  CRISIS MAE is {ratio:.1f}x higher than LOW MAE')
    print('  → Motivates conformal prediction intervals in Phase 6')

fig.add_vline(x=0, line_color='black', line_dash='dash', line_width=2)
fig.update_layout(
    title='LSTM Residuals by Market Regime (t+5)',
    xaxis_title='Residual (Predicted - Actual) VIX pts',
    yaxis_title='Density',
    barmode='overlay',
    height=420
)
fig.write_image('figures/lstm_residuals_by_regime.png', width=1100, height=420, scale=1)
del fig
gc.collect()
print('\nSaved: figures/lstm_residuals_by_regime.png')

RESIDUAL ANALYSIS BY REGIME (t+5)
  LOW       : n= 628 | MAE=4.1046 | RMSE=4.9066 | Bias=+3.4569
  ELEVATED  : n= 100 | MAE=5.2358 | RMSE=7.2733 | Bias=+1.3545
  CRISIS    : n=  13 | MAE=6.9936 | RMSE=8.5277 | Bias=-6.2461

  CRISIS MAE is 1.7x higher than LOW MAE
  → Motivates conformal prediction intervals in Phase 6

Saved: figures/lstm_residuals_by_regime.png


## Cell 13 - Rolling 60-Day MAE 

A single overall MAE number (e.g. 2.1 VIX pts) hides temporal variation in model performance. Rolling 60-day MAE computes the metric on a sliding window across the entire test period.

In [17]:
import gc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
gc.collect()

ROLL_WINDOW = 60
abs_errors  = np.abs(predicted_5 - actual_5)
rolling_mae = pd.Series(abs_errors, index=dates).rolling(ROLL_WINDOW).mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=[f'Rolling {ROLL_WINDOW}-Day MAE', 'Actual VIX'],
                    vertical_spacing=0.12)

fig.add_trace(go.Scatter(x=dates, y=rolling_mae.values,
              name=f'{ROLL_WINDOW}-day MAE',
              line=dict(color='tomato', width=1.8)), row=1, col=1)
fig.add_hline(y=mae5,   line_dash='dash', line_color='steelblue',
              annotation_text=f'Overall MAE: {mae5:.4f}', row=1, col=1)
fig.add_hline(y=2.2295, line_dash='dot',  line_color='orange',
              annotation_text='Persistence: 2.2295',       row=1, col=1)

fig.add_trace(go.Scatter(x=dates, y=actual_5,
              name='Actual VIX',
              line=dict(color='black', width=1.2)), row=2, col=1)
fig.add_hline(y=20, line_dash='dot', line_color='orange', row=2, col=1)
fig.add_hline(y=30, line_dash='dot', line_color='red',    row=2, col=1)

fig.update_yaxes(title_text='MAE (VIX pts)', row=1, col=1)
fig.update_yaxes(title_text='VIX',           row=2, col=1)
fig.update_layout(title=f'Rolling {ROLL_WINDOW}-Day MAE - Test Period Stability', height=580)
fig.write_image('figures/lstm_rolling_mae.png', width=1200, height=580, scale=1)
del fig
gc.collect()

valid = rolling_mae.dropna()
print('Saved: figures/lstm_rolling_mae.png')
print(f'Worst 60-day window : {valid.idxmax().date()}  MAE={valid.max():.4f}')
print(f'Best  60-day window : {valid.idxmin().date()}  MAE={valid.min():.4f}')

Saved: figures/lstm_rolling_mae.png
Worst 60-day window : 2025-06-24  MAE=10.2748
Best  60-day window : 2023-10-23  MAE=1.6367


## Cell 14 - Attention Weight Visualisation 

The attention layer assigns a weight to each of the 60 input timesteps. A perfectly uniform model would assign weight `1/60 ≈ 0.017` to every timestep , meaning it treats all days in the window equally. Weights significantly above 0.017 reveal which positions in the 60-day window the model found most informative.

In [18]:
import gc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
gc.collect()

mean_attn   = attn_weights.mean(axis=0)
uniform     = 1.0 / SEQUENCE_LENGTH
crisis_mask = regime_arr[:n] == 'CRISIS'
low_mask    = regime_arr[:n] == 'LOW'
crisis_attn = attn_weights[crisis_mask].mean(axis=0) if crisis_mask.sum() > 0 else mean_attn
low_attn    = attn_weights[low_mask].mean(axis=0)    if low_mask.sum() > 0    else mean_attn

positions   = list(range(SEQUENCE_LENGTH))
tick_pos    = list(range(0, SEQUENCE_LENGTH, 10))
tick_labels = [f't-{SEQUENCE_LENGTH - 1 - i}' for i in tick_pos]
bar_colors  = ['crimson' if w > uniform * 1.5 else 'steelblue' for w in mean_attn]
timesteps   = [f't-{SEQUENCE_LENGTH - 1 - i}' for i in range(SEQUENCE_LENGTH)]

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Mean Attention - all test samples (red=high)',
                                    'LOW vs CRISIS Attention'])

fig.add_trace(go.Bar(x=positions, y=list(mean_attn),
              marker_color=bar_colors, name='Mean attention',
              showlegend=False), row=1, col=1)
fig.add_hline(y=uniform, line_dash='dash', line_color='red',
              annotation_text=f'Uniform ({uniform:.4f})', row=1, col=1)

fig.add_trace(go.Scatter(x=positions, y=list(low_attn),
              name='LOW', line=dict(color='steelblue', width=2)), row=1, col=2)
fig.add_trace(go.Scatter(x=positions, y=list(crisis_attn),
              name='CRISIS', line=dict(color='crimson', width=2)), row=1, col=2)
fig.add_hline(y=uniform, line_dash='dash', line_color='grey', row=1, col=2)

fig.update_xaxes(tickvals=tick_pos, ticktext=tick_labels, tickangle=45)
fig.update_yaxes(title_text='Attention Weight', row=1, col=1)
fig.update_layout(title='Attention Weights Over 60-Day Input Window', height=420)
fig.write_image('figures/lstm_attention_weights.png', width=1200, height=420, scale=1)
del fig
gc.collect()
print('Saved: figures/lstm_attention_weights.png')

top3 = np.argsort(mean_attn)[::-1][:3]
print('Top 3 attended timesteps:')
for i in top3:
    print(f'  {timesteps[i]}: {mean_attn[i]:.5f}  ({mean_attn[i]/uniform:.1f}x uniform)')

Saved: figures/lstm_attention_weights.png
Top 3 attended timesteps:
  t-0: 0.01692  (1.0x uniform)
  t-1: 0.01692  (1.0x uniform)
  t-2: 0.01691  (1.0x uniform)


## Cell 15 - Update Results Table


In [19]:
results_path = 'models/results_table.csv'

if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
else:
    # Recreate Phase 4 baselines if CSV was not carried forward
    results_df = pd.DataFrame([
        {'Model': 'Persistence (Naive)', 'MAE': 2.2295, 'RMSE': None,
         'MAPE': None, 'Dir_Accuracy': 53.03, 'Notes': 'True floor — must beat this'},
        {'Model': 'ARIMA(1,1,1)',        'MAE': 2.2580, 'RMSE': None,
         'MAPE': None, 'Dir_Accuracy': 53.55, 'Notes': 'Univariate, walk-forward CV'},
        {'Model': 'GARCH(1,1)',          'MAE': None,   'RMSE': None,
         'MAPE': None, 'Dir_Accuracy': 60.38, 'Notes': 'Volatility model, dir acc only'},
    ])

# Remove any existing LSTM row (safe to re-run)
results_df = results_df[~results_df['Model'].str.contains('LSTM', na=False)].copy()

r5 = results[5]
lstm_row = pd.DataFrame([{
    'Model'        : 'Multivariate LSTM + Attention',
    'MAE'          : round(r5['mae'],     4),
    'RMSE'         : round(r5['rmse'],    4),
    'MAPE'         : round(r5['mape'],    2),
    'Dir_Accuracy' : round(r5['dir_acc'], 2),
    'Notes'        : (f'seq={SEQUENCE_LENGTH}, hidden={HIDDEN_SIZE}, '
                      f'layers={NUM_LAYERS}, dropout={DROPOUT}, '
                      f'attn=True, horizons={HORIZONS}, '
                      f'best_ep={best_epoch}, seed={SEED}')
}])

results_df = pd.concat([results_df, lstm_row], ignore_index=True)
results_df.to_csv(results_path, index=False)

print('Updated: models/results_table.csv')
print()
print(results_df.to_string(index=False))

KeyError: 'Model'